## 1. Imports + Config

# Notebook 01 — Data Build

## Objective

Construct the canonical balancing-market modelling dataset
by combining:

- BM balancing actions (BOALF),
- balancing prices (BOA),
- system prices,
- and engineered settlement-period timestamps.

The notebook creates the foundational datasets used
throughout the later fair-value and adaptive-equilibrium analysis.

Primary outputs:

- bm_actions.parquet
- modelling_dataset_enriched.parquet

## 1. Imports and Configuration

This section loads the core Python libraries used for:
- data engineering,
- timestamp handling,
- numerical operations,
- and downstream persistence.

A processed-data directory is also created to support
reproducible parquet outputs.

In [1]:
import pandas as pd
import numpy as np
import requests
import statsmodels.api as sm
import matplotlib.pyplot as plt

import os

os.makedirs("../data/processed", exist_ok=True)

print("Processed directory created")

Processed directory created


## 2. Load Core Datasets

The project combines multiple BMRS datasets:

| Dataset | Purpose |
|---|---|
| BOALF | Accepted balancing actions and volumes |
| BOA | Accepted balancing prices |
| System prices | System-wide balancing reference prices |
| Modelling dataset | Wind, demand, and settlement-period features |

These datasets are loaded separately before being
standardised and merged.

In [2]:
df = pd.read_csv("../data/modelling_dataset_2022.csv")
boalf = pd.read_csv("../data/boalf_sw_2022.csv")
boa = pd.read_csv("../data/boa_2022.csv")
price = pd.read_csv("../data/system_prices_2022.csv")

In [3]:
# boalf = pd.read_csv("../data/boalf_sw_2022.csv")

modelling = pd.read_csv(
    "../data/modelling_dataset_2022.csv"
)

system_price = pd.read_csv(
    "../data/system_prices_2022.csv"
)

## 3. Standardise Column Names

BMRS datasets use inconsistent naming conventions.

This section standardises:
- settlement-period fields,
- BM unit identifiers,
- pricing variables,
- and timestamp columns

to simplify downstream joins and aggregation logic.

In [4]:
boalf = boalf.rename(columns={
    'settlementDate': 'settlement_date',
    'settlementPeriodFrom': 'settlement_period',
    'nationalGridBmUnit': 'bm_unit',
    'acceptanceNumber': 'acceptance_number',
    'levelFrom': 'level_from',
    'levelTo': 'level_to'
})

boa = boa.rename(columns={
    'settlementDate': 'settlement_date',
    'settlementPeriod': 'settlement_period',
    'bmUnit': 'bm_unit',
    'acceptanceNumber': 'acceptance_number',
    'bidPrice': 'bid_price',
    'offerPrice': 'offer_price'
})

price = price.rename(columns={
    'settlementDate': 'date',
    'settlementPeriod': 'sp',
    'system_price_gbp_mwh': 'system_price'
})

## 4. Construct Settlement-Period Timestamps

BMRS data is indexed by:
- settlement date,
- and settlement period.

A continuous timestamp is constructed by converting
settlement periods into 30-minute offsets.

This creates a common temporal key used across all datasets.

In [5]:
df['date'] = pd.to_datetime(df['date'])
price['date'] = pd.to_datetime(price['date'])
boalf['settlement_date'] = pd.to_datetime(boalf['settlement_date'])

df['timestamp'] = (
    df['date'] +
    pd.to_timedelta((df['sp'] - 1) * 30, unit='m')
)

price['timestamp'] = (
    price['date'] +
    pd.to_timedelta((price['sp'] - 1) * 30, unit='m')
)

boalf['timestamp'] = (
    boalf['settlement_date'] +
    pd.to_timedelta((boalf['settlement_period'] - 1) * 30, unit='m')
)

## 5. Merge BOA Prices Into BOALF Actions

BOALF contains accepted balancing actions and volumes,
while BOA contains associated bid and offer prices.

These datasets are merged using acceptance numbers
to create a unified balancing-action dataset.

In [6]:
bm = boalf.merge(
    boa[
        [
            'acceptance_number',
            'bid_price',
            'offer_price'
        ]
    ],
    on='acceptance_number',
    how='left'
)

## 6. Construct Balancing VWAP

Accepted balancing actions are converted into:
- signed accepted MW,
- directional bid/offer pricing,
- and volume-weighted average balancing prices (VWAP).

Negative accepted MW values represent bids
(downward balancing actions),
while positive values represent offers
(upward balancing actions).

The notebook constructs a settlement-period VWAP
using accepted balancing volumes and prices.

In [7]:
bm['accepted_mw'] = bm['level_to'] - bm['level_from']

bm = bm[bm['accepted_mw'] != 0]

bm['accepted_price'] = np.where(
    bm['accepted_mw'] < 0,
    bm['bid_price'],
    bm['offer_price']
)

bm_bids = bm[bm['accepted_mw'] < 0].copy()

bm_bids = bm_bids.dropna(subset=['accepted_price'])

bm_bids['volume_mw'] = bm_bids['accepted_mw'].abs()

bm_bids = bm_bids[bm_bids['volume_mw'] >= 1]

bm_vwap = (
    bm_bids
    .assign(value=lambda x: x['volume_mw'] * x['accepted_price'])
    .groupby('timestamp')
    .agg(
        total_volume_mw=('volume_mw', 'sum'),
        total_value=('value', 'sum')
    )
    .reset_index()
)

bm_vwap['bm_bid_price_vwap'] = (
    bm_vwap['total_value'] / bm_vwap['total_volume_mw']
)

### Important Note

This VWAP construction represents an operational balancing-price proxy
derived from accepted balancing actions.

It should not be interpreted as a fully tradable market price,
but rather as an aggregated measure of accepted balancing activity
within each settlement period.

## 7. Merge System Prices and Balancing VWAP

System prices and balancing VWAP values are merged into
the main modelling dataset using settlement-period timestamps.

This creates the basis for:
- spread construction,
- fair-value estimation,
- and later adaptive-equilibrium modelling.

In [8]:
df = df.merge(
    price[['timestamp', 'system_price']],
    on='timestamp',
    how='left'
)

df = df.merge(
    bm_vwap[['timestamp', 'bm_bid_price_vwap']],
    on='timestamp',
    how='left'
)

## 8. Create Final Modelling Variables

This section constructs the key modelling variables used
throughout the project.

Key variables include:

| Variable | Description |
|---|---|
| basis_spread | System price minus balancing VWAP |
| relative_spread | Spread normalised by system price |
| stress | High-price stress indicator |
| hour | Intraday timing feature |

These variables form the foundation for the later
market-state and fair-value analysis notebooks.

In [9]:
df['basis_spread'] = (
    df['system_price'] - df['bm_bid_price_vwap']
)

df_valid = df.dropna(subset=['basis_spread']).copy()

df_valid['relative_spread'] = (
    df_valid['basis_spread'] / df_valid['system_price']
)

df_valid['hour'] = df_valid['timestamp'].dt.hour

df_valid['stress'] = (
    df_valid['system_price'] > 300
).astype(int)

In [10]:
df['timestamp'] = pd.to_datetime(df['timestamp'])

In [15]:
df.to_parquet(
    "../data/processed/modelling_dataset_enriched.parquet",
    index=False
)

bm.to_parquet(
    "../data/processed/bm_actions.parquet",
    index=False
)

# Notebook Summary

This notebook constructed the canonical balancing-market dataset used
throughout the project.

Key outputs:
- timestamp-aligned BM actions,
- balancing VWAP estimates,
- system-price spreads,
- and engineered market-state variables.

The resulting parquet datasets provide the foundation for:
- exploratory balancing analysis,
- volatility clustering analysis,
- fair-value estimation,
- and adaptive equilibrium modelling in later notebooks.